In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, 
                             confusion_matrix,
                             f1_score, recall_score,
                             roc_auc_score)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import mlflow
import joblib
import os

print("✅ All libraries imported!")
print(f"XGBoost version: {xgb.__version__}")

✅ All libraries imported!
XGBoost version: 3.2.0


In [2]:
# Load engineered features from Phase 2
features = pd.read_csv('../data/processed/features.csv')

print("✅ Features loaded!")
print(f"Shape: {features.shape}")
print(f"\nClass distribution:")
print(features['FLAG'].value_counts())
print(f"\nTheft %: {round(features['FLAG'].mean()*100, 2)}%")

✅ Features loaded!
Shape: (42372, 20)

Class distribution:
FLAG
0    38757
1     3615
Name: count, dtype: int64

Theft %: 8.53%


In [3]:
# Separate features and target
X = features.drop(['CONS_NO', 'FLAG'], axis=1)
y = features['FLAG']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature names: {list(X.columns)}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% for testing
    stratify=y,         # Keep same theft ratio in both sets
    random_state=42     # Reproducible results
)

print(f"\n✅ Split done!")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")
print(f"\nTrain theft %: {round(y_train.mean()*100, 2)}%")
print(f"Test theft %:  {round(y_test.mean()*100, 2)}%")

Features shape: (42372, 18)
Target shape: (42372,)
Feature names: ['mean_consumption', 'std_consumption', 'max_consumption', 'min_consumption', 'median_consumption', 'skewness', 'kurtosis', 'zero_reading_rate', 'negative_count', 'missing_rate', 'coeff_variation', 'low_reading_rate', 'first_half_mean', 'second_half_mean', 'consumption_trend', 'p25', 'p75', 'iqr']

✅ Split done!
Training samples: 33897
Testing samples:  8475

Train theft %: 8.53%
Test theft %:  8.53%


In [5]:
from sklearn.impute import SimpleImputer

# Fill NaN values with median of each feature
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

# Convert back to DataFrame
X_train_imp = pd.DataFrame(X_train_imp, columns=X.columns)
X_test_imp  = pd.DataFrame(X_test_imp,  columns=X.columns)

print("✅ Missing values filled!")
print(f"NaN in train: {X_train_imp.isnull().sum().sum()}")
print(f"NaN in test:  {X_test_imp.isnull().sum().sum()}")

✅ Missing values filled!
NaN in train: 0
NaN in test:  0


In [6]:
# Now apply SMOTE on clean data
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_imp, y_train)

print("✅ SMOTE applied!")
print(f"\nBefore SMOTE:")
print(f"  Normal: {(y_train==0).sum()}")
print(f"  Theft:  {(y_train==1).sum()}")
print(f"\nAfter SMOTE:")
print(f"  Normal: {(y_train_bal==0).sum()}")
print(f"  Theft:  {(y_train_bal==1).sum()}")

✅ SMOTE applied!

Before SMOTE:
  Normal: 31005
  Theft:  2892

After SMOTE:
  Normal: 31005
  Theft:  15502


In [7]:
print("Training Logistic Regression...")

lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr.fit(X_train_bal, y_train_bal)

y_pred_lr = lr.predict(X_test_imp)

print("✅ Logistic Regression Done!")
print("\n--- Results ---")
print(classification_report(y_test, y_pred_lr,
      target_names=['Normal','Theft']))
print(f"ROC-AUC: {round(roc_auc_score(y_test, lr.predict_proba(X_test_imp)[:,1]),4)}")

Training Logistic Regression...
✅ Logistic Regression Done!

--- Results ---
              precision    recall  f1-score   support

      Normal       0.94      0.81      0.87      7752
       Theft       0.19      0.49      0.27       723

    accuracy                           0.78      8475
   macro avg       0.57      0.65      0.57      8475
weighted avg       0.88      0.78      0.82      8475

ROC-AUC: 0.7207


In [8]:
print("Training Random Forest...")

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_bal, y_train_bal)

y_pred_rf = rf.predict(X_test_imp)

print("✅ Random Forest Done!")
print("\n--- Results ---")
print(classification_report(y_test, y_pred_rf,
      target_names=['Normal','Theft']))
print(f"ROC-AUC: {round(roc_auc_score(y_test, rf.predict_proba(X_test_imp)[:,1]),4)}")

Training Random Forest...
✅ Random Forest Done!

--- Results ---
              precision    recall  f1-score   support

      Normal       0.93      0.96      0.95      7752
       Theft       0.33      0.19      0.24       723

    accuracy                           0.90      8475
   macro avg       0.63      0.58      0.59      8475
weighted avg       0.88      0.90      0.89      8475

ROC-AUC: 0.7343


In [9]:
print("Training XGBoost...")

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=11,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)
xgb_model.fit(X_train_bal, y_train_bal)

y_pred_xgb = xgb_model.predict(X_test_imp)

print("✅ XGBoost Done!")
print("\n--- Results ---")
print(classification_report(y_test, y_pred_xgb,
      target_names=['Normal','Theft']))
print(f"ROC-AUC: {round(roc_auc_score(y_test, xgb_model.predict_proba(X_test_imp)[:,1]),4)}")

Training XGBoost...
✅ XGBoost Done!

--- Results ---
              precision    recall  f1-score   support

      Normal       0.96      0.56      0.71      7752
       Theft       0.14      0.74      0.23       723

    accuracy                           0.58      8475
   macro avg       0.55      0.65      0.47      8475
weighted avg       0.89      0.58      0.67      8475

ROC-AUC: 0.7299


In [10]:
print("Training Tuned XGBoost...")

xgb_tuned = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    scale_pos_weight=11,
    gamma=0.1,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

xgb_tuned.fit(
    X_train_bal, y_train_bal,
    eval_set=[(X_test_imp, y_test)],
    verbose=False
)

y_pred_tuned = xgb_tuned.predict(X_test_imp)

print("✅ Tuned XGBoost Done!")
print("\n--- Results ---")
print(classification_report(y_test, y_pred_tuned,
      target_names=['Normal','Theft']))
print(f"ROC-AUC: {round(roc_auc_score(y_test, xgb_tuned.predict_proba(X_test_imp)[:,1]),4)}")

Training Tuned XGBoost...
✅ Tuned XGBoost Done!

--- Results ---
              precision    recall  f1-score   support

      Normal       0.97      0.44      0.60      7752
       Theft       0.12      0.85      0.21       723

    accuracy                           0.47      8475
   macro avg       0.55      0.64      0.41      8475
weighted avg       0.90      0.47      0.57      8475

ROC-AUC: 0.7467


In [12]:
# Instead of default 0.5 threshold, try 0.3
# Lower threshold = catches more thieves
from sklearn.metrics import precision_score
y_proba_tuned = xgb_tuned.predict_proba(X_test_imp)[:,1]

print("Testing different thresholds:")
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<10} {'F1':<8}")
print("-" * 44)

for threshold in [0.2, 0.3, 0.4, 0.5, 0.6]:
    y_pred_t = (y_proba_tuned >= threshold).astype(int)
    p = round(precision_score(y_test, y_pred_t, zero_division=0), 3)
    r = round(recall_score(y_test, y_pred_t), 3)
    f = round(f1_score(y_test, y_pred_t), 3)
    print(f"{threshold:<12} {p:<12} {r:<10} {f:<8}")

Testing different thresholds:
Threshold    Precision    Recall     F1      
--------------------------------------------
0.2          0.098        0.963      0.178   
0.3          0.105        0.941      0.189   
0.4          0.114        0.905      0.202   
0.5          0.123        0.846      0.214   
0.6          0.137        0.763      0.232   
